# Homework 4: Evaluation

Questions: [homework.md](../../course/cohorts/2026/homework/04-evaluation/homework.md)

## Setup

In [4]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [5]:
len(documents)

72

In [6]:
print(documents[0]["content"][:50] + "...")
print(documents[0]["filename"])

# Introduction

Video: [Watch this lesson](https:/...
01-agentic-rag/lessons/01-intro.md


## Generating ground truth

In [7]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

## Q1. Generating questions

In [8]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [9]:
import json

user_prompt = json.dumps(documents[0])

In [10]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [11]:
# pydantic is like zod
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [12]:
response = openai_client.responses.parse(
    model="gpt-6-luna",
    input=messages,
    text_format=Questions
)

In [13]:
result = response.output_parsed

print(result.questions)

['What does a large language model do when you give it a prompt?', 'Why might an LLM give a wrong answer or lack information about your documents?', 'How does RAG help an LLM answer questions using information it wasn’t trained on?', 'What will the FAQ agent being built in this module do?', 'How does the agentic version of the pipeline differ from the fixed workflow in Part 1?']


In [14]:
from evaluation_utils import llm_structured

def generate_questions(doc):
      user_prompt = json.dumps(doc)

      result, usage = llm_structured(
          openai_client,
          data_gen_instructions,
          user_prompt,
          Questions
      )

      return result, usage

In [15]:
result, usage = generate_questions(documents[0])
usage

ResponseUsage(input_tokens=1020, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0, cache_write_tokens_details={'ttl_12h_tokens': 0, 'ttl_30m_tokens': 0}), output_tokens=151, output_tokens_details=OutputTokensDetails(reasoning_tokens=58), total_tokens=1171)

In [16]:
result, usage = generate_questions(documents[1])
usage

ResponseUsage(input_tokens=1286, input_tokens_details=InputTokensDetails(cache_write_tokens=1283, cached_tokens=0, cache_write_tokens_details={'ttl_12h_tokens': 0, 'ttl_30m_tokens': 1283}), output_tokens=152, output_tokens_details=OutputTokensDetails(reasoning_tokens=52), total_tokens=1438)

In [17]:
result, usage = generate_questions(documents[2])
usage

ResponseUsage(input_tokens=1753, input_tokens_details=InputTokensDetails(cache_write_tokens=1750, cached_tokens=0, cache_write_tokens_details={'ttl_12h_tokens': 0, 'ttl_30m_tokens': 1750}), output_tokens=167, output_tokens_details=OutputTokensDetails(reasoning_tokens=66), total_tokens=1920)

## The full ground truth

In [18]:
import pandas as pd

df_ground_truth = pd.read_csv("ground-truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

len(ground_truth)

360

## Searching the chunks

In [19]:

from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

len(chunks)

295

In [20]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=lambda key: scores[key], reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [21]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(chunks)


In [23]:
def text_search(query, num_results):
  return index.search(query, num_results)

In [ ]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

## Q2. First result with text search

In [ ]:
q = ground_truth[0]["question"]

NameError: name 'ground_truth' is not defined

## Q3. First result with vector search

## Evaluation metrics

## Q4. Evaluating text search

## Q5. Evaluating vector search

## Q6. Tuning hybrid search